## 1. Setup and Imports

In [1]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.2 MB/s eta 0:00:00


In [2]:
import os
import random
import pandas as pd
import sentencepiece as spm
import sacrebleu
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import zipfile
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.9.0+cu126
CUDA available: True


In [3]:
!gdown 1KuDLpUaSe0wzDWEhgG0wKXV1_AVmzLK4

Downloading...
From: https://drive.google.com/uc?id=1KuDLpUaSe0wzDWEhgG0wKXV1_AVmzLK4
To: /content/dataset.zip
100% 928k/928k [00:00<00:00, 10.2MB/s]


In [4]:
!unzip dataset.zip

Archive:  dataset.zip
   creating: dataset/
   creating: dataset/private_test/
  inflating: dataset/private_test/private_test.zh  
   creating: dataset/public_test/
  inflating: dataset/public_test/public_test.zh  
   creating: dataset/train/
  inflating: dataset/train/train.vi  
  inflating: dataset/train/train.zh  


## 2. Configuration and Hyperparameters

In [5]:
# Data paths
TRAIN_SRC = "dataset/train/train.zh"
TRAIN_TGT = "dataset/train/train.vi"
TEST_SRC = "dataset/public_test/public_test.zh"
PRIVATE_SRC = "dataset/private_test/private_test.zh"
SAVE_DIR = "./checkpoints"
SPM_ZH_PREFIX = os.path.join(SAVE_DIR, "spm_zh")
SPM_VI_PREFIX = os.path.join(SAVE_DIR, "spm_vi")

# Model hyperparameters
VOCAB_SIZE = 3000
EMB_SIZE = 64
HID_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 10
LR = 0.01
MAX_LEN = 80
SEED = 42

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Set random seeds for reproducibility
os.makedirs(SAVE_DIR, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

Using device: cuda


## 3. Data Loading and Preprocessing

In [6]:
def read_lines(path):
    """Read lines from a text file."""
    with open(path, "r", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

# Load training and test data
train_src = read_lines(TRAIN_SRC)
train_tgt = read_lines(TRAIN_TGT)
test_src = read_lines(TEST_SRC)
private_src = read_lines(PRIVATE_SRC)

print(f"Training samples: {len(train_src)}")
print(f"Test samples: {len(test_src)}")
print(f"\nExample Chinese sentence: {train_src[0]}")
print(f"Example Vietnamese sentence: {train_tgt[0]}")

Training samples: 32061
Test samples: 1781

Example Chinese sentence: 我 会 给 您 拿 一些 。
Example Vietnamese sentence: Tôi sẽ mang cho bạn một_ít .


## 4. SentencePiece Tokenization

Train BPE tokenizers for both Chinese and Vietnamese.

In [7]:
def train_spm(input_file, model_prefix, vocab_size=VOCAB_SIZE):
    """Train a SentencePiece BPE model."""
    args = (
        f"--input={input_file} --model_prefix={model_prefix} --vocab_size={vocab_size} "
        "--model_type=bpe --character_coverage=1.0 "
        "--pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3"
    )
    spm.SentencePieceTrainer.Train(args)
    print(f"Trained SentencePiece model: {model_prefix}.model")

def load_sp(model_path):
    """Load a trained SentencePiece model."""
    sp = spm.SentencePieceProcessor()
    sp.Load(model_path)
    return sp

In [8]:
# Train Chinese tokenizer
tmp_zh = os.path.join(SAVE_DIR, "tmp_zh.txt")
if not os.path.exists(SPM_ZH_PREFIX + ".model"):
    with open(tmp_zh, "w", encoding="utf-8") as f:
        for s in train_src:
            f.write(s + "\n")
    train_spm(tmp_zh, SPM_ZH_PREFIX)

# Train Vietnamese tokenizer
tmp_vi = os.path.join(SAVE_DIR, "tmp_vi.txt")
if not os.path.exists(SPM_VI_PREFIX + ".model"):
    with open(tmp_vi, "w", encoding="utf-8") as f:
        for s in train_tgt:
            f.write(s + "\n")
    train_spm(tmp_vi, SPM_VI_PREFIX)

# Load tokenizers
sp_zh = load_sp(SPM_ZH_PREFIX + ".model")
sp_vi = load_sp(SPM_VI_PREFIX + ".model")

print(f"\nChinese vocab size: {sp_zh.GetPieceSize()}")
print(f"Vietnamese vocab size: {sp_vi.GetPieceSize()}")

# Test tokenization
test_sent = train_src[0]
tokens = sp_zh.EncodeAsIds(test_sent)
print(f"\nExample tokenization:")
print(f"Original: {test_sent}")
print(f"Token IDs: {tokens[:20]}...")

Trained SentencePiece model: ./checkpoints/spm_zh.model
Trained SentencePiece model: ./checkpoints/spm_vi.model

Chinese vocab size: 3000
Vietnamese vocab size: 3000

Example tokenization:
Original: 我 会 给 您 拿 一些 。
Token IDs: [5, 47, 28, 56, 109, 162, 4]...


## 5. Dataset and DataLoader

In [9]:
class TranslationDataset(Dataset):
    """Dataset for Chinese-Vietnamese translation pairs."""

    def __init__(self, src, tgt, sp_src, sp_tgt, max_len=MAX_LEN):
        self.src = src
        self.tgt = tgt
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        # Add BOS (2) and EOS (3) tokens
        src_ids = [2] + self.sp_src.EncodeAsIds(self.src[idx])[:self.max_len-2] + [3]
        tgt_ids = [2] + self.sp_tgt.EncodeAsIds(self.tgt[idx])[:self.max_len-2] + [3]
        return torch.tensor(src_ids), torch.tensor(tgt_ids)

def collate_fn(batch):
    """Collate function to pad sequences to the same length."""
    srcs, tgts = zip(*batch)
    max_src = max(len(s) for s in srcs)
    max_tgt = max(len(t) for t in tgts)

    # Pad with 0 (PAD token)
    src_pad = torch.zeros(len(batch), max_src, dtype=torch.long)
    tgt_pad = torch.zeros(len(batch), max_tgt, dtype=torch.long)

    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, :len(s)] = s
        tgt_pad[i, :len(t)] = t

    return src_pad, tgt_pad

In [10]:
# Create training dataset and dataloader
# Split data: 90% train, 10% validation
total_samples = len(train_src)
train_size = int(0.9 * total_samples)

train_src_split = train_src[:train_size]
train_tgt_split = train_tgt[:train_size]
valid_src = train_src[train_size:]
valid_tgt = train_tgt[train_size:]

print(f"Total samples: {total_samples}")
print(f"Training samples: {len(train_src_split)}")
print(f"Validation samples: {len(valid_src)}")

dataset = TranslationDataset(train_src_split, train_tgt_split, sp_zh, sp_vi)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True)

valid_dataset = TranslationDataset(valid_src, valid_tgt, sp_zh, sp_vi)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f"Training batches: {len(dataloader)}")
print(f"Validation batches: {len(valid_loader)}")

Total samples: 32061
Training samples: 28854
Validation samples: 3207
Training batches: 451
Validation batches: 101


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


## 6. Model Architecture

### Encoder-Decoder with GRU

In [11]:
class EncoderRNN(nn.Module):
    """GRU-based encoder."""

    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=0)
        self.dropout = nn.Dropout(0.3)
        self.rnn = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, src):
        emb = self.dropout(self.embedding(src))
        _, hidden = self.rnn(emb)
        return hidden


class DecoderRNN(nn.Module):
    """GRU-based decoder with teacher forcing."""

    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=0)
        self.dropout = nn.Dropout(0.3)
        self.rnn = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_step, hidden):
        emb = self.dropout(self.embedding(input_step))
        output, hidden = self.rnn(emb, hidden)
        pred = self.fc(output.squeeze(1))
        return pred, hidden


class Seq2Seq(nn.Module):
    """Sequence-to-sequence model."""

    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.3):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        vocab_size = self.decoder.fc.out_features

        # Store outputs
        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(src.device)

        # Encode source sentence
        hidden = self.encoder(src)

        # Start with BOS token
        input_step = tgt[:, 0].unsqueeze(1)

        # Decode step by step
        for t in range(1, tgt_len):
            pred, hidden = self.decoder(input_step, hidden)
            outputs[:, t] = pred

            # Teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            input_step = tgt[:, t].unsqueeze(1) if teacher_force else pred.argmax(1).unsqueeze(1)

        return outputs

In [12]:
# Initialize model
encoder = EncoderRNN(sp_zh.GetPieceSize(), EMB_SIZE, HID_SIZE)
decoder = DecoderRNN(sp_vi.GetPieceSize(), EMB_SIZE, HID_SIZE)
model = Seq2Seq(encoder, decoder).to(DEVICE)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model has {count_parameters(model):,} trainable parameters")
print(f"\nModel architecture:")
print(model)

Model has 919,992 trainable parameters

Model architecture:
Seq2Seq(
  (encoder): EncoderRNN(
    (embedding): Embedding(3000, 64, padding_idx=0)
    (dropout): Dropout(p=0.3, inplace=False)
    (rnn): GRU(64, 128, batch_first=True)
  )
  (decoder): DecoderRNN(
    (embedding): Embedding(3000, 64, padding_idx=0)
    (dropout): Dropout(p=0.3, inplace=False)
    (rnn): GRU(64, 128, batch_first=True)
    (fc): Linear(in_features=128, out_features=3000, bias=True)
  )
)


## 7. Training Functions

In [13]:
def train_epoch(model, dataloader, criterion, optimizer):
    """Train for one epoch."""
    model.train()
    total_loss = 0

    for src, tgt in dataloader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        optimizer.zero_grad()
        output = model(src, tgt)

        # Calculate loss (ignore first BOS token)
        loss = criterion(
            output[:, 1:].reshape(-1, output.size(-1)),
            tgt[:, 1:].reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate_bleu(model, dataloader, sp_tgt):
    """Evaluate model using SacreBLEU metric."""
    model.eval()
    hyps, refs = [], []
    pbar = tqdm(dataloader, desc="Evaluating", leave=False)
    for src, tgt in pbar:
        src = src.to(DEVICE)
        # Encode
        hidden = model.encoder(src)
        # Decode (greedy)
        input_step = torch.full((src.size(0), 1), 2, dtype=torch.long, device=DEVICE)
        decoded = [[] for _ in range(src.size(0))]
        for _ in range(MAX_LEN):
            pred, hidden = model.decoder(input_step, hidden)
            next_token = pred.argmax(1).unsqueeze(1)
            for i in range(src.size(0)):
                decoded[i].append(next_token[i].item())
            input_step = next_token
        # Convert to text
        for i in range(src.size(0)):
            ids = decoded[i]
            if 3 in ids:  # Stop at EOS
                ids = ids[:ids.index(3)]
            hyps.append(sp_tgt.DecodeIds(ids))
            ref_ids = tgt[i].tolist()[1:-1]  # Remove BOS and EOS
            refs.append(sp_tgt.DecodeIds([x for x in ref_ids if x not in [0, 1]]))
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    return bleu.score

## 8. Training Loop

In [14]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Starting training...\n")

# Training loop with best model saving
best_bleu = 0.0
best_model_path = os.path.join(SAVE_DIR, "best_model.pt")

for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, dataloader, criterion, optimizer)
    bleu = evaluate_bleu(model, valid_loader, sp_vi)

    # Save best model
    if bleu > best_bleu:
        best_bleu = bleu
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'bleu': bleu,
            'loss': loss
        }, best_model_path)
        print(f"Epoch {epoch:02d} | Loss={loss:.3f} | SacreBLEU={bleu:.2f} Best model saved!")
    else:
        print(f"Epoch {epoch:02d} | Loss={loss:.3f} | SacreBLEU={bleu:.2f}")

print(f"\nTraining completed!")
print(f"Best validation BLEU: {best_bleu:.2f}")
print(f"Best model saved to: {best_model_path}")

# Load best model for inference
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']}")

Starting training...



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch 01 | Loss=5.060 | SacreBLEU=2.38 Best model saved!


Epoch 02 | Loss=4.507 | SacreBLEU=3.26 Best model saved!


Epoch 03 | Loss=4.260 | SacreBLEU=3.69 Best model saved!


...

Epoch 06 | Loss=3.919 | SacreBLEU=4.38


Epoch 07 | Loss=3.847 | SacreBLEU=5.21 Best model saved!


Epoch 08 | Loss=3.795 | SacreBLEU=5.36 Best model saved!


Epoch 09 | Loss=3.755 | SacreBLEU=5.50 Best model saved!


Epoch 10 | Loss=3.724 | SacreBLEU=5.27

Training completed!
Best validation BLEU: 5.50
Best model saved to: ./checkpoints/best_model.pt
Loaded best model from epoch 9


## 9. Translation on Test Set

In [15]:
@torch.no_grad()
def translate_test(model, test_src, sp_src, sp_tgt, out_path):
    """Translate test set and save to CSV."""
    model.eval()
    outputs = []

    for s in tqdm(test_src, desc="Translating"):
        # Tokenize source sentence
        src_ids = [2] + sp_src.EncodeAsIds(s)[:MAX_LEN-2] + [3]
        src_tensor = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

        # Encode
        hidden = model.encoder(src_tensor)

        # Decode
        input_step = torch.full((1, 1), 2, dtype=torch.long, device=DEVICE)
        decoded = []

        for _ in range(MAX_LEN):
            pred, hidden = model.decoder(input_step, hidden)
            next_token = pred.argmax(1)
            token = next_token.item()

            if token == 3:  # EOS
                break

            decoded.append(token)
            input_step = next_token.unsqueeze(1)

        # Decode to Vietnamese text
        vi_sent = sp_tgt.DecodeIds(decoded)
        outputs.append(vi_sent)

    # Save to CSV
    df = pd.DataFrame({"tieng_trung": test_src, "tieng_viet": outputs})
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    return out_path

In [16]:
public_submission_path = os.path.join("public_test.csv")
translate_test(model, test_src, sp_zh, sp_vi, public_submission_path)

print(f"Translation completed!")
print(f"Saved to: {public_submission_path}")

# Display sample translations
df_result = pd.read_csv(public_submission_path)
print("\nSample translations:")
print(df_result.head(10))

Translating: 100%|██████████| 1781/1781 [00:08<00:00, 210.66it/s]

Translation completed!
Saved to: public_test.csv

Sample translations:
                              tieng_trung  \
0   就 我 来说 , 通常 都 是 经营 的 问题 ， 很 少 带来 欢乐 。   
1                          你 有 哪些 运动 器材 ？   
2                          酒店 有 会议 设施 吗 ？   
3                        我 会 度 过 这 段 时间 。   
4                        我 需要 一 张 填表 谢谢 。   
5                             我 有 慢性 哮喘 。   
6                         坐 船 过 池塘 要 多久 ？   
7                          她 想 要 一 杯 啤酒 。   
8                                  去 纽约 。   
9  你 今晚 八点半 左右 不 是 偶然 在 十二 G 街 的 拐角 是 吗 ？   

                                          tieng_viet  
0  Theoưa tôi gọi gì tôi là gì , tôi________________  
1                                    Bạn có_______ ?  
2                            Có có_______sạn không ?  
3                    Tôi nghĩ tôi biết kết_hônQuốc .  
4  Làm vuiơn cho tôi một một đôi đế thuốc với phấ...  
5                                 Tôi có____________  
6                                Đi_____________

In [17]:
private_submission_path = os.path.join("private_test.csv")
translate_test(model, private_src, sp_zh, sp_vi, private_submission_path)

print(f"Translation completed!")
print(f"Saved to: {private_submission_path}")

df_result = pd.read_csv(private_submission_path)
print("\nSample translations:")
print(df_result.head(10))

Translating: 100%|██████████| 1781/1781 [00:08<00:00, 216.24it/s]

Translation completed!
Saved to: private_test.csv

Sample translations:
                                    tieng_trung  \
0                                   我 会 呆 两 天 。   
1                                   我 现在 在 机场 。   
2                                 玛丽 不 比 亨利 大 。   
3                                 这些 问题 并不 新鲜 。   
4                              再 说 一 次 你 的 姓名 。   
5                                我 大概 有 三千 美元 。   
6                     当然 了 先生 , 可是 你 出 什么 事 了 ？   
7                                     你 需要 活动 。   
8                                   它 在 星期几 开 ？   
9  你 知道 日本 的 摔跤 传统 , 在 相扑 冠军 之间 有 一 个 是 美国人 吗 ？   

                                          tieng_viet  
0                  Tôi dự_định đặt phòng ngày ngày .  
1                                   Tôi ngồi________  
2                          Cácạy_đất không__________  
3    Món này này phim này là những gìcách thiết_kế .  
4                          Có , nói bạn bạn nói bạn_  
5                   Tôi trăm đô ba t

## 10. Create Submission ZIP

In [18]:
# Create ZIP file for submission
zip_path = os.path.join("submission.zip")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(public_submission_path, arcname="public_test.csv")
    zipf.write(private_submission_path, arcname="private_test.csv")

print(f"Submission ZIP created: {zip_path}")
print(f"File size: {os.path.getsize(zip_path) / 1024:.2f} KB")

Submission ZIP created: submission.zip
File size: 99.28 KB
